In [1]:
from dotenv import load_dotenv
import json
import os
from pathlib import Path
from pydub import AudioSegment
import shutil, errno
import pandas as pd
from elevenlabs.client import ElevenLabs
from elevenlabs import play
from io import BytesIO
from elevenlabs.client import ElevenLabs

In [49]:
def extension(file):
    return Path(file).suffix.replace('.', '')

In [4]:
load_dotenv('openai.env')

True

In [38]:
file = 'data/Kate Rich DJNF profile.m4a'
output_name = Path(file).stem.replace(' ', '_')

In [6]:
elevenlabs = ElevenLabs(
  api_key=os.getenv("ELEVENLABS_API_KEY"),
)

In [15]:
with open(file, 'rb') as in_file:
    raw_audio = BytesIO(in_file.read())

In [16]:
transcription = elevenlabs.speech_to_text.convert(
    file=raw_audio,
    model_id="scribe_v1", # Model to use, for now only "scribe_v1" is supported
    tag_audio_events=True, # Tag audio events like laughter, applause, etc.
    language_code="eng", # Language of the audio file. If set to None, the model will detect the language automatically.
    diarize=True, # Whether to annotate who is speaking
)

In [21]:
transcript = dict(transcription)

In [50]:
transcript

{'language_code': 'eng',
 'language_probability': 1.0,
 'text': 'All right, so yeah, I\'m writing a profile of Levy. Uh, she\'s one of the other interns. I understand you guys have worked together in the past. Yeah. Yeah, so she\'s our communications intern. Uh, this is her last week, and we\'re really gonna miss her. And I\'m the Communications Specialist here at eScience. Awesome. Uh, can you tell me a little bit more about the projects you\'ve worked on or that you\'ve gotten to see her work on? Yeah. So, Levy was here for a number of things. Um, kind of the first thing she was hired on to do, uh, was to help with the formatting, uh, and the written content of our newsletter. Mm-hmm. We have a weekly newsletter or bulletin that we send out, uh, that\'s largely about data science and AI content coming out of the eScience Institute. Mm-hmm. Um, Levy was such a perfect and unique fit for this, because I, I think it\'s very hard sometimes to find someone who likes writing, knows how to 

In [51]:
result = []
i = 0
prev_speaker = dict(transcript['words'][0])['speaker_id']
result.append({
            'speaker_id': prev_speaker,
            'text': '',
            'words': []
        })
for word in transcript['words']:
    word = dict(word)
    curr_speaker = word['speaker_id']
    if curr_speaker == prev_speaker:
        result[i]['text'] += word['text']
        result[i]['words'].append(word)
    else:
        i += 1
        result.append({
            'speaker_id': curr_speaker,
            'text': word['text'],
            'words': [word]
        })
    prev_speaker = curr_speaker

In [53]:
result = {
    'segments': result
}

In [54]:
with open(f'{output_name}_transcript.json', 'w') as out_path:  
    json.dump(result, out_path)